In [366]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler, MinMaxScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, root_mean_squared_error, root_mean_squared_log_error


In [417]:
pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None)

In [368]:
#log

import logging

logging.basicConfig(
    filename='app.log',
    filemode='a',
    level=logging.DEBUG,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s', 
    datefmt='%Y-%m-%d %H:%M:%S',  
)

logger = logging.getLogger()



## Load Data

In [418]:
house = pd.read_csv("house_price_prediction_dataset.csv").drop(["Id"], axis=1)
house.head(5)

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,1stFlrSF,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,TotRmsAbvGrd,Functional,Fireplaces,FireplaceQu,GarageType,GarageYrBlt,GarageFinish,GarageCars,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,Y,SBrkr,856,854,0,1710,1,0,2,1,3,1,Gd,8,Typ,0,NaN,Attchd,2003.0,RFn,2,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,Y,SBrkr,1262,0,0,1262,0,1,2,0,3,1,TA,6,Typ,1,TA,Attchd,1976.0,RFn,2,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,Y,SBrkr,920,866,0,1786,1,0,2,1,3,1,Gd,6,Typ,1,TA,Attchd,2001.0,RFn,2,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,Y,SBrkr,961,756,0,1717,1,0,1,0,3,1,Gd,7,Typ,1,Gd,Detchd,1998.0,Unf,3,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,Y,SBrkr,1145,1053,0,2198,1,0,2,1,4,1,Gd,9,Typ,1,TA,Attchd,2000.0,RFn,3,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000


## Basic Analysis

- Understand the size of the data
- Basic info of the dataset
- Check if there is any missing features in the dataset
- capture unique values of categorical and numerical columns and write to files
- describe the numerical features in the dataset

In [419]:
object_columns = house.select_dtypes('object').columns.to_list()
numerical_columns = house.select_dtypes('number').columns.to_list()

In [420]:
def check_missing_values(df):
    missing_feature_counts = df.isna().sum()
    missing_feature_df = (missing_feature_counts[missing_feature_counts >0].to_frame()).reset_index()

    #print(f"total number of missing features: {missing_feature_df.shape[0]}")
    missing_feature_df.columns = ["Feature", "Count"]

    missing_feature_df["null_percentage"] = round((missing_feature_df["Count"]/df.shape[0]) *100, 2)
    missing_feature_df["Type"] = missing_feature_df["Feature"].map(df.dtypes)
    
    return missing_feature_df.sort_values(by="null_percentage", ascending= False)

In [421]:
def get_dataset_details(df):
    
    object_columns = df.select_dtypes('object').columns.to_list()
    numerical_columns = df.select_dtypes('number').columns.to_list()
    
    missing_features = check_missing_values(df)

    with open ("object_columns.txt", 'w') as file:
        for obj_col in object_columns:
            #file.writelines(f"{obj_col}:\n")
            file.write(f"{obj_col} Feature contains {house[obj_col].nunique()} unique values:\n \n")
            file.write(f"{house[obj_col].value_counts().to_string()} \n \n")
    
    with open ("numerical_columns.txt", 'w') as file:
        for num_col in numerical_columns:
            #file.writelines(f"{obj_col}:\n")
            file.write(f"{num_col} Feature contains {house[num_col].nunique()} unique values:\n \n")
            if house[num_col].nunique() <= 10:
                file.write(f"{house[num_col].value_counts().to_string()} \n \n")

    return {"object_columns": object_columns,
            "numerical_columns": numerical_columns,
            "missing_features": missing_features}


In [422]:
def highly_correlated_features(data, threshold):
    corr_matrix = data.corr().abs().round(2)
    upper_triangle = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k =1).astype(bool))
    high_corr_features = [ (col, row) for col in upper_triangle.columns for row in upper_triangle.index if upper_triangle[col][row] > threshold]
    return high_corr_features

## EDA

In [423]:
def eda(data):
    sample, features = data.shape
    #print(f"The house price dataset contains {sample} samples and {features} features")
    data_deatils = get_dataset_details(data)

    object_columns = data_deatils["object_columns"]
    numerical_columns = data_deatils["numerical_columns"]
    missing_features = data_deatils["missing_features"]

    skewness = data[numerical_columns].skew().sort_values(ascending=False) # using third moment about the mean
    highly_skewed_columns = skewness[(skewness > 1) | (skewness < -1)].index.to_list()
    #mod_skewed_columns = skewness[(skewness > 1) or (skewness <-1) ].index.to_list()

    kurtosis = data[numerical_columns].kurtosis().sort_values(ascending=False)
    leptokurtic = kurtosis[kurtosis > 0].index.to_list() # heavy tail, higher probability of extreme values

    #correlation
    threshold = 0.7
    high_correlated_features = highly_correlated_features(data[numerical_columns], threshold)


    logger.info(f"The house price dataset contains {sample} samples and {features} features\n")
    logger.info(f"Total Number of object colums: {len(object_columns)}\n")
    logger.info(f"Total Number of Numerical colums: {len(numerical_columns)}\n")
    logger.info(f"Total Number of Missing Features:\n {len(missing_features)}\n")
    logger.info(f"Missing Features:\n {missing_features}\n")
    logger.info("===" * 30)

    logger.info("\n Numerical column summary:\n")
    logger.info(f"{(house.describe().T).to_string()}\n")

    logger.info("===" * 30)
    logger.info("\n Skewness:\n")
    logger.info(skewness.to_string())

    logger.info(f"\n\n No of highly Skewed Features: {len(highly_skewed_columns)}\n")
    logger.info(f"\n Highly Skewed Features: {highly_skewed_columns}\n")

    logger.info("===" * 30)
    logger.info("\n excess Kustosis:\n")
    logger.info(f"count= {len(leptokurtic)},\nFeatures= {leptokurtic}\n")

    logger.info("===" * 30)
    logger.info(f"\n Highly correlated features based on threshold {threshold}:\n {high_correlated_features}")
    
    
    print(f"Total Number of object colums: {len(object_columns)}")
    print(f"Total Number of Numerical colums: {len(numerical_columns)}")
    print(f"Total Number of Missing Features: {len(missing_features)}\n")
    #print(f"Missing Features:\n {missing_features}\n")
    print(f"\n No of highly Skewed Features: {len(highly_skewed_columns)}\n")
    print(f"\n Highly Skewed Features: {highly_skewed_columns}\n")

    print(f"Heavy Kustosis: count= {len(leptokurtic)},\nFeatures= {leptokurtic}")

    print(f"Highly correlated features based on threshold {threshold}: {high_correlated_features}")
    
    # print("===" * 50)

    # print("Numerical column summary")
    # print(house.describe().T)



In [424]:
eda(house)

Total Number of object colums: 43
Total Number of Numerical colums: 37
Total Number of Missing Features: 19


 No of highly Skewed Features: 20


 Highly Skewed Features: ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'BsmtHalfBath', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'LotFrontage', 'SalePrice', 'BsmtFinSF1', 'WoodDeckSF', 'TotalBsmtSF', 'MSSubClass', '1stFlrSF', 'GrLivArea']

Heavy Kustosis: count= 27,
Features= ['MiscVal', 'PoolArea', 'LotArea', '3SsnPorch', 'LowQualFinSF', 'KitchenAbvGr', 'BsmtFinSF2', 'ScreenPorch', 'LotFrontage', 'BsmtHalfBath', 'TotalBsmtSF', 'BsmtFinSF1', 'EnclosedPorch', 'MasVnrArea', 'OpenPorchSF', 'SalePrice', '1stFlrSF', 'GrLivArea', 'WoodDeckSF', 'BedroomAbvGr', 'MSSubClass', 'OverallCond', 'GarageArea', 'TotRmsAbvGrd', 'BsmtUnfSF', 'GarageCars', 'OverallQual']
Highly correlated features based on threshold 0.7: [('1stFlrSF', 'TotalBsmtSF'), ('TotRmsAbvGrd', 'GrLivArea'), ('GarageYrBlt'

# Univariate Analysis

- Analyze the distribution of individual features
- Outlier Identification
- Missing value treatment

In [376]:
# def hist_plot(data):
#     no_features = data.shape[1]
#     no_cols = 4
#     no_rows = (no_features // 4) +1

#     fig, axes = plt.subplots(no_rows, no_cols, figsize=(24,no_rows * 6))
#     axes = axes.flatten()

#     for i, col in enumerate(data.columns.to_list()):
#         feature_mean = data[col].mean().round(2)
#         feature_median = data[col].median()
#         #print(feature_mean, feature_median)

#         axes[i].hist(data[col], bins = 20, edgecolor='k')
#         axes[i].axvline(feature_mean, color = 'red', linestyle="--", label=f"Mean:{feature_mean}")
#         axes[i].axvline(feature_median, color='green', linestyle = "-", label = f"Median:{feature_median}")

#         axes[i].set_title(f"Distribution of '{col}'")
#         axes[i].set_xlabel(col)
#         axes[i].set_ylabel("Frequency")
    
#     for j in range(no_features, len(axes)):
#         axes[j].set_visible(False)


In [377]:
# def box_plot(data):
#     no_features = data.shape[1]
#     no_cols = 4
#     no_rows = (no_features // 4) +1

#     fig, axes = plt.subplots(no_rows, no_cols, figsize=(24,no_rows * 6))
#     axes = axes.flatten()

#     for i, col in enumerate(data.columns.to_list()):
#         sns.boxplot(data[col], color='lightblue', ax=axes[i])
#         #axes[i].hist(data[col], bins = 20, edgecolor='k')
#         axes[i].set_title(f"Boxplot of '{col}'")
#         # axes[i].set_xlabel(col)
#         # axes[i].set_ylabel("Frequency")
    
#     for j in range(no_features, len(axes)):
#         axes[j].set_visible(False)


In [378]:
def Numerical_column_distribution(data):
    no_features = data.shape[1]

    fig, axes = plt.subplots(no_features, 2, figsize=(14, no_features * 6))
    
    #axes = axes.flatten()

    for i, col in enumerate(data.columns.to_list()):

        feature_mean = data[col].mean().round(2)
        feature_median = data[col].median()
        #print(feature_mean, feature_median)

        sns.histplot(data[col], bins = 20, edgecolor='k', kde=True, ax=axes[i][0])
        axes[i][0].axvline(feature_mean, color = 'red', linestyle="--", label=f"Mean:{feature_mean}")
        axes[i][0].axvline(feature_median, color='green', linestyle = "-", label = f"Median:{feature_median}")

        axes[i][0].set_title(f"Distribution of '{col}'")
        axes[i][0].set_xlabel(col)
        axes[i][0].set_ylabel("Frequency")


        sns.boxplot(data[col], color='lightblue', ax=axes[i][1])
        #axes[i].hist(data[col], bins = 20, edgecolor='k')
        axes[i][1].set_title(f"Boxplot of '{col}'")
        axes[i][1].set_xlabel(col)
        # axes[i].set_ylabel("Frequency")
    
    for j in range(no_features, len(axes)):
        axes[j].set_visible(False)

## Numerical Column Distribution

In [379]:


#Numerical_column_distribution(house[numerical_columns])

## Categorical Column Distribution

In [380]:
def categorical_column_distribution(data):
    no_features = data.shape[1]
    no_cols = 2
    no_rows = no_features//no_cols +1 
    
    print(no_features)
    fig, axes = plt.subplots(no_rows, no_cols, figsize = (16,60))
    axes = axes.flatten()

    for i, col in enumerate(data.columns.to_list()):
        value_counts = data[col].value_counts()
        #print(value_counts)
        axes[i].bar(value_counts.index, value_counts.values)

    for j in range(no_features, len(axes)):
        axes[j].set_visible(False)
    

In [381]:
#categorical_column_distribution(house[object_columns])

# Feature Engineering

The data contains 1460 samples and 79 features, No of features are quite high. 
We will use Filter methods to determine, 
- low variance fetures from each independent features
- compare variance between groups vs. variance within group - find F-statistics
- MI (Mutual information) - dependency between feature X and target Y

question - if I add new feature derived from existing should it be done after preprocessing. that means after splitting, morever we nwould need scaling of those features as well, so should not it be done before importance and preprocessing

In [431]:
(house.var(numeric_only=True)).round(3).sort_values(ascending=False)


SalePrice        6.311111e+09
LotArea          9.962565e+07
GrLivArea        2.761296e+05
MiscVal          2.461381e+05
BsmtFinSF1       2.080255e+05
BsmtUnfSF        1.952464e+05
TotalBsmtSF      1.924624e+05
2ndFlrSF         1.905571e+05
1stFlrSF         1.494501e+05
GarageArea       4.571251e+04
MasVnrArea       3.278497e+04
BsmtFinSF2       2.602391e+04
WoodDeckSF       1.570981e+04
OpenPorchSF      4.389861e+03
EnclosedPorch    3.735550e+03
ScreenPorch      3.108889e+03
LowQualFinSF     2.364204e+03
MSSubClass       1.789338e+03
PoolArea         1.614216e+03
YearBuilt        9.122150e+02
3SsnPorch        8.595060e+02
GarageYrBlt      6.095830e+02
LotFrontage      5.897490e+02
YearRemodAdd     4.262330e+02
MoSold           7.310000e+00
TotRmsAbvGrd     2.642000e+00
OverallQual      1.913000e+00
YrSold           1.764000e+00
OverallCond      1.238000e+00
BedroomAbvGr     6.650000e-01
GarageCars       5.580000e-01
Fireplaces       4.160000e-01
FullBath         3.040000e-01
BsmtFullBa

# Data Cleanup

In [382]:
 #### get columns which has null value count more that 50%
missing_feature_counts = check_missing_values(house)
cols_to_drop = missing_feature_counts[missing_feature_counts["null_percentage"] > 40]["Feature"].to_list()
logger.info(f"Features which contain more than 40% null value: {cols_to_drop}")
#house.drop(cols_to_drop, axis=1, inplace=True)

#hightly corrlated features - ('1stFlrSF', 'TotalBsmtSF'), ('TotRmsAbvGrd', 'GrLivArea'), ('GarageYrBlt', 'YearBuilt'), ('GarageArea', 'GarageCars')

#house.drop(["1stFlrSF", "TotRmsAbvGrd", "GarageYrBlt", "GarageCars"], axis=1, inplace= True)

In [383]:
def data_cleanup(df):
    # encode 'CentralAir' as boolean, 

    
    df['CentralAir'] = df['CentralAir'].map({'Y': 1, 'N': 0})

   
    df['HouseAge'] = df['YrSold'] - df['YearBuilt']
    df['HouseRemodAge'] = df['YrSold']  - df['YearRemodAdd']

    df["TotalSF"] = (df["TotalBsmtSF"] +df["1stFlrSF"] + df["2ndFlrSF"])

    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"] + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])
    
    df.drop(["1stFlrSF", "TotRmsAbvGrd", "GarageYrBlt", "GarageCars"], axis=1, inplace= True)
    

    return df

In [384]:
cleaned_df = data_cleanup(house)
cleaned_df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,Functional,Fireplaces,FireplaceQu,GarageType,GarageFinish,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HouseAge,HouseRemodAge,TotalSF,TotalBath
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,1,SBrkr,854,0,1710,1,0,2,1,3,1,Gd,Typ,0,NaN,Attchd,RFn,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500,5,5,2566,3.5
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,1,SBrkr,0,0,1262,0,1,2,0,3,1,TA,Typ,1,TA,Attchd,RFn,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500,31,31,2524,2.5
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,1,SBrkr,866,0,1786,1,0,2,1,3,1,Gd,Typ,1,TA,Attchd,RFn,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500,7,6,2706,3.5
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,1,SBrkr,756,0,1717,1,0,1,0,3,1,Gd,Typ,1,Gd,Detchd,Unf,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000,91,36,2473,2.0
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,1,SBrkr,1053,0,2198,1,0,2,1,4,1,Gd,Typ,1,TA,Attchd,RFn,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000,8,8,3343,3.5


In [385]:
cleaned_df.shape

(1460, 80)

In [386]:
# As per random forest model below features does not have any importance, So drop them from the feature list
#"Electrical","Heating_Wall", "Functional", "RoofMatl", "RoofStyle", "SaleType_ConLw", "GarageType_Basment", "SaleCondition_Family", 
#"HouseStyle", "Condition2", "Neighborhood", "LotConfig_Inside", "Utilities", "Street","MSZoning", "YrSold","MiscVal", 
#"3SsnPorch", "EnclosedPorch","KitchenAbvGr", "BsmtHalfBath", "BsmtFullBath", "LowQualFinSF", "BldgType", "Condition1", "PavedDrive"


# cleaned_df.drop(["Electrical","Heating", "Functional", "RoofMatl", "RoofStyle", "SaleType", "GarageType", 
#                  "SaleCondition", "HouseStyle", "Condition2", "Neighborhood", "LotConfig", "Utilities", "Street",
#                  "MSZoning","MiscVal", "3SsnPorch", "EnclosedPorch","KitchenAbvGr", "BsmtHalfBath", "BsmtFullBath", 
#                  "LowQualFinSF", "BldgType", "Condition1", "PavedDrive", "MoSold"], inplace=True, axis=1)



In [387]:
cleaned_df.shape

(1460, 80)

In [388]:
cat_cols = cleaned_df.select_dtypes(include="object").columns.to_list()
num_cols = cleaned_df.select_dtypes(exclude="object").columns.to_list()
print(cat_cols)
print(num_cols)

['MSZoning', 'Street', 'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig', 'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType', 'HouseStyle', 'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'Heating', 'HeatingQC', 'Electrical', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'PavedDrive', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType', 'SaleCondition']
['MSSubClass', 'LotFrontage', 'LotArea', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'CentralAir', '2ndFlrSF', 'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath', 'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'Fireplaces', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch', '3SsnPorch', 'ScreenPorch', 'PoolArea', 'MiscVa

# Preprocessing +Feature Selection PCA

In [389]:
cleaned_df.head()

,MSSubClass,MSZoning,LotFrontage,LotArea,Street,Alley,LotShape,LandContour,Utilities,LotConfig,LandSlope,Neighborhood,Condition1,Condition2,BldgType,HouseStyle,OverallQual,OverallCond,YearBuilt,YearRemodAdd,RoofStyle,RoofMatl,Exterior1st,Exterior2nd,MasVnrType,MasVnrArea,ExterQual,ExterCond,Foundation,BsmtQual,BsmtCond,BsmtExposure,BsmtFinType1,BsmtFinSF1,BsmtFinType2,BsmtFinSF2,BsmtUnfSF,TotalBsmtSF,Heating,HeatingQC,CentralAir,Electrical,2ndFlrSF,LowQualFinSF,GrLivArea,BsmtFullBath,BsmtHalfBath,FullBath,HalfBath,BedroomAbvGr,KitchenAbvGr,KitchenQual,Functional,Fireplaces,FireplaceQu,GarageType,GarageFinish,GarageArea,GarageQual,GarageCond,PavedDrive,WoodDeckSF,OpenPorchSF,EnclosedPorch,3SsnPorch,ScreenPorch,PoolArea,PoolQC,Fence,MiscFeature,MiscVal,MoSold,YrSold,SaleType,SaleCondition,SalePrice,HouseAge,HouseRemodAge,TotalSF,TotalBath
0,60,RL,65.0,8450,Pave,NaN,Reg,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2003,2003,Gable,CompShg,VinylSd,VinylSd,BrkFace,196.0,Gd,TA,PConc,Gd,TA,No,GLQ,706,Unf,0,150,856,GasA,Ex,1,SBrkr,854,0,1710,1,0,2,1,3,1,Gd,Typ,0,NaN,Attchd,RFn,548,TA,TA,Y,0,61,0,0,0,0,NaN,NaN,NaN,0,2,2008,WD,Normal,208500,5,5,2566,3.5
1,20,RL,80.0,9600,Pave,NaN,Reg,Lvl,AllPub,FR2,Gtl,Veenker,Feedr,Norm,1Fam,1Story,6,8,1976,1976,Gable,CompShg,MetalSd,MetalSd,NaN,0.0,TA,TA,CBlock,Gd,TA,Gd,ALQ,978,Unf,0,284,1262,GasA,Ex,1,SBrkr,0,0,1262,0,1,2,0,3,1,TA,Typ,1,TA,Attchd,RFn,460,TA,TA,Y,298,0,0,0,0,0,NaN,NaN,NaN,0,5,2007,WD,Normal,181500,31,31,2524,2.5
2,60,RL,68.0,11250,Pave,NaN,IR1,Lvl,AllPub,Inside,Gtl,CollgCr,Norm,Norm,1Fam,2Story,7,5,2001,2002,Gable,CompShg,VinylSd,VinylSd,BrkFace,162.0,Gd,TA,PConc,Gd,TA,Mn,GLQ,486,Unf,0,434,920,GasA,Ex,1,SBrkr,866,0,1786,1,0,2,1,3,1,Gd,Typ,1,TA,Attchd,RFn,608,TA,TA,Y,0,42,0,0,0,0,NaN,NaN,NaN,0,9,2008,WD,Normal,223500,7,6,2706,3.5
3,70,RL,60.0,9550,Pave,NaN,IR1,Lvl,AllPub,Corner,Gtl,Crawfor,Norm,Norm,1Fam,2Story,7,5,1915,1970,Gable,CompShg,Wd Sdng,Wd Shng,NaN,0.0,TA,TA,BrkTil,TA,Gd,No,ALQ,216,Unf,0,540,756,GasA,Gd,1,SBrkr,756,0,1717,1,0,1,0,3,1,Gd,Typ,1,Gd,Detchd,Unf,642,TA,TA,Y,0,35,272,0,0,0,NaN,NaN,NaN,0,2,2006,WD,Abnorml,140000,91,36,2473,2.0
4,60,RL,84.0,14260,Pave,NaN,IR1,Lvl,AllPub,FR2,Gtl,NoRidge,Norm,Norm,1Fam,2Story,8,5,2000,2000,Gable,CompShg,VinylSd,VinylSd,BrkFace,350.0,Gd,TA,PConc,Gd,TA,Av,GLQ,655,Unf,0,490,1145,GasA,Ex,1,SBrkr,1053,0,2198,1,0,2,1,4,1,Gd,Typ,1,TA,Attchd,RFn,836,TA,TA,Y,192,84,0,0,0,0,NaN,NaN,NaN,0,12,2008,WD,Normal,250000,8,8,3343,3.5


In [390]:
from sklearn.impute import SimpleImputer


X = cleaned_df.drop(["SalePrice"], axis=1) 
y = cleaned_df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

logger.info(f"Total features: {X_train.shape[1]}")
cat_cols = X_train.select_dtypes(include="object").columns.to_list()
num_cols = X_train.select_dtypes(exclude="object").columns.to_list()

logger.info(f"Categorical_Columns: {cat_cols}")
logger.info(f"Numerical Columns: {num_cols}")

logger.info("==" * 30)

logger.info("Update column list after randomforest feature importance")
# 'CentralAir' - already encoded as boolean, 

cat_onehotencode = ['MSZoning','Street', 'Utilities','LotConfig', 'Neighborhood', 'Condition1','Condition2', 'BldgType', 'HouseStyle',\
                    'RoofStyle', 'RoofMatl', 'Heating', 'Electrical' , 'GarageType', 'SaleType', 'SaleCondition' ]  #review Utilities as it contains all same value

cat_ordinal = ['LotShape', 'LandContour', 'LandSlope', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',\
               'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual' ,'Functional', 'GarageFinish', \
                'GarageQual', 'GarageCond', 'PavedDrive']


#cat_onehotencode = ['MSSubClass']

# num_cols = ['LotFrontage', 'LotArea', 'YearBuilt', 'YearRemodAdd', 'MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 
#             'TotalBsmtSF', '1stFlrSF', '2ndFlrSF', 'GrLivArea', 'GarageArea', 'WoodDeckSF', 'OpenPorchSF', 'ScreenPorch', 
#             'PoolArea', 'TotalSF', 'TotalBath']

# cat_ordinal = ['LotShape', 'LandContour', 'LandSlope', 'Exterior1st', 'Exterior2nd', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',\
#                'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'HeatingQC', 'KitchenQual' ,'GarageFinish', \
#                 'GarageQual', 'GarageCond',]

# pass_cols= ['MSSubClass', 'OverallQual', 'OverallCond', 'CentralAir', 'FullBath','HalfBath', 'BedroomAbvGr', 'TotRmsAbvGrd', 'Fireplaces', 
#             'GarageYrBlt', 'GarageCars','YrSold','HouseAge', 'HouseRemodAge', 'TotalBath']



numeric_transformer = Pipeline(
    steps=[
        #('mean_imputer', SimpleImputer(strategy='mean')),
        ('median_imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler())
        ]
)

categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)



preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal),
    #('pass', SimpleImputer(strategy='median'), pass_cols)
])



# # include PCA

pca = PCA(n_components=0.95)
pipeline_pca = Pipeline(
    steps=[
        ('preprocess', preprocessor),
        ('PCA', pca)
    ]
)

# logger.info("==" * 30)
# logger.info("preprocess the data using PCA")
# X_train_scaled_pca = pipeline_pca.fit_transform(X_train)
# X_test_scaled_pca = pipeline_pca.transform(X_test)

# logger.info(f"After transformation shape of X_train scaled: {X_train_scaled_pca.shape}")
# logger.info(f"After transformation shape of X_test scaled: {X_test_scaled_pca.shape}")

# logger.info(f"Explained variance ratio:\n {pca.explained_variance_ratio_}")
# logger.info(f"Total Explained variance ratio: {sum(pca.explained_variance_ratio_)}")



# ## without pca
# logger.info("==" * 30)
# logger.info("preprocess the data without PCA")
# X_train_scaled = preprocessor.fit_transform(X_train)
# X_test_scaled = preprocessor.transform(X_test)

# logger.info(f"After transformation shape of X_train scaled: {X_train_scaled.shape}")
# logger.info(f"After transformation shape of X_test scaled: {X_test_scaled.shape}")

# feature_names =[name.split("__")[1] for name in preprocessor.get_feature_names_out()]
# logger.info(f"Preprocesed FeatureNames: {feature_names}")




In [391]:
pipeline_pca.get_feature_names_out

<bound method Pipeline.get_feature_names_out of Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('num',
                                                  Pipeline(steps=[('median_imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['MSSubClass', 'LotFrontage',
                                                   'LotArea', 'OverallQual',
                                                   'OverallCond', 'YearBuilt',
                                                   'YearRemodAdd', 'MasVnrArea',
                                                   'BsmtFinSF1', 'BsmtFinSF2',
                                                   'BsmtUnfSF', 'TotalBsmtSF',
                                     

# Baseline model

## Linear Regression

In [392]:
def create_model(preprocessor, model):
    reggresor = Pipeline(
    steps=[
        ('pre', preprocessor),
        ("lr_lasso_pca", model)
    ])
    return reggresor

In [393]:
from sklearn.linear_model import Ridge, Lasso

alpha = 0.1

model_pipelines = [("linear_regression", create_model(preprocessor, LinearRegression(tol=0.0001, fit_intercept=True))), 
                   ("linear_regression_pca", create_model(pipeline_pca, LinearRegression(tol=0.0001, fit_intercept=True))),
                   ("linear_regression_ridge", create_model(preprocessor, Ridge(tol=0.0001, alpha=alpha, fit_intercept=True))),
                   ("linear_regression_ridge_pca", create_model(pipeline_pca, Ridge(tol=0.0001, alpha=alpha, fit_intercept=True))),
                   ("linear_regression_lasso", create_model(preprocessor, Lasso(tol=0.0001, alpha=alpha, fit_intercept=True))),
                   ("linear_regression_lasso_pca", create_model(pipeline_pca, Lasso(tol=0.0001, alpha=alpha, fit_intercept=True)))] 


r2_scores = []

for name, pipe in model_pipelines:
    pipe.fit(X_train, y_train)
    y_test_pred = pipe.predict(X_test)
    y_train_pred = pipe.predict(X_train)


    r2_score_test = r2_score(y_test,y_test_pred)
    r2_score_train = r2_score(y_train, y_train_pred)

    rmse_test = root_mean_squared_error(y_test,y_test_pred)
    rmse_train = root_mean_squared_error(y_train, y_train_pred)  

    r2_scores.append({
                "Model": name, 
                "R2_score-Test": r2_score_test, 
                "R2_score-Train": r2_score_train,
                "RMSE-Test": rmse_test, 
                "RMSE-Train": rmse_train
                })  



    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML

,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795


## Decision Tree

In [394]:
from sklearn.tree import DecisionTreeRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

preprocessor_decision_tree = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal),
    #('pass', SimpleImputer(strategy='median'), pass_cols)
])

decision_tree_regrssor = Pipeline(
    steps=[
        ("pre", preprocessor_decision_tree),
        ("decision_tree", DecisionTreeRegressor(min_samples_split=20,random_state=42))
    ]
)

model_decision_tree = decision_tree_regrssor.fit(X_train, y_train)

y_test_pred = model_decision_tree.predict(X_test)
y_train_pred = model_decision_tree.predict(X_train)


r2_score_test = r2_score(y_test,y_test_pred)
r2_score_train = r2_score(y_train, y_train_pred)

rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

r2_scores.append({
            "Model": "Decision Tree", 
            "R2_score-Test": r2_score_test, 
            "R2_score-Train": r2_score_train,
            "RMSLE-Test": rmse_test, 
            "RMSLE-Train": rmse_train
            })  
    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466,NaN,NaN
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786,NaN,NaN
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273,NaN,NaN
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910,NaN,NaN
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680,NaN,NaN
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795,NaN,NaN
6,Decision Tree,0.792529,0.934768,NaN,NaN,0.193743,0.102175


In [395]:
model_decision_tree

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('pre', ...), ('decision_tree', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat_onehotencode', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the differen

In [396]:

feature_names_dt =[name.split("__")[1] for name in preprocessor_decision_tree.get_feature_names_out()]
logger.info(f"Decision Tree Preprocesed FeatureNames: {feature_names_dt}")

feature_importance_dt = model_decision_tree["decision_tree"].feature_importances_.round(3)
feature_importance_df = pd.DataFrame({
                            "Feature": feature_names_dt, "importance": feature_importance_dt
                            }).sort_values(by='importance', ascending= False)

high_imp_features = feature_importance_df[feature_importance_df['importance'] >0]
#model_decision_tree_pca.feature_importances_.round(3)

logger.info("Feature Importance as per Decision Tree")
logger.info(high_imp_features.to_string())

print("Feature Importance as per Decision Tree")
print(high_imp_features.to_string())

Feature Importance as per Decision Tree
                  Feature  importance
3             OverallQual       0.540
35                TotalSF       0.295
13               2ndFlrSF       0.031
36              TotalBath       0.022
1             LotFrontage       0.020
6            YearRemodAdd       0.012
23             GarageArea       0.010
8              BsmtFinSF1       0.010
2                 LotArea       0.006
11            TotalBsmtSF       0.006
15              GrLivArea       0.005
31                 MoSold       0.005
22             Fireplaces       0.005
10              BsmtUnfSF       0.004
136              LotShape       0.004
117     GarageType_Attchd       0.002
7              MasVnrArea       0.002
5               YearBuilt       0.002
141             ExterQual       0.002
12             CentralAir       0.002
139           Exterior1st       0.001
140           Exterior2nd       0.001
52   Neighborhood_Crawfor       0.001
26          EnclosedPorch       0.001
151       

# Advance Regression

### Random Forest

In [397]:
from sklearn.ensemble import RandomForestRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

rf_preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal),
    #('pass', SimpleImputer(strategy='median'), pass_cols)
])

random_forest_regressor = Pipeline(
    steps=[
        ("pre", rf_preprocessor),
        ("random_forest", RandomForestRegressor(n_estimators=500, max_depth=8, min_samples_split=20, random_state=42))
    ]
)

random_forest_regressor.fit(X_train, y_train)
y_test_pred = random_forest_regressor.predict(X_test)
y_train_pred = random_forest_regressor.predict(X_train)


r2_score_test = r2_score(y_test,y_test_pred)
r2_score_train = r2_score(y_train, y_train_pred)

rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

r2_scores.append({
            "Model": "Random Forst Regressor", 
            "R2_score-Test": r2_score_test, 
            "R2_score-Train": r2_score_train,
            "RMSLE-Test": rmse_test, 
            "RMSLE-Train": rmse_train
            })  

pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466,NaN,NaN
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786,NaN,NaN
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273,NaN,NaN
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910,NaN,NaN
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680,NaN,NaN
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795,NaN,NaN
6,Decision Tree,0.792529,0.934768,NaN,NaN,0.193743,0.102175
7,Random Forst Regressor,0.866345,0.935190,NaN,NaN,0.160718,0.104405


In [398]:

feature_names_dt =[name.split("__")[1] for name in preprocessor_decision_tree.get_feature_names_out()]
logger.info(f"Random Forest Tree Preprocesed FeatureNames: {feature_names_dt}")

feature_importance_rf = random_forest_regressor["random_forest"].feature_importances_.round(3)
feature_importance_df = pd.DataFrame({
                            "Feature": feature_names_dt, "importance_dt": feature_importance_dt, 
                            "Feature": feature_names_dt, "importance_rf": feature_importance_rf
                            }).sort_values(by='importance_rf', ascending= False)

high_imp_features = feature_importance_df[(feature_importance_df['importance_dt'] >0) | (feature_importance_df['importance_rf'] >0)]
#model_decision_tree_pca.feature_importances_.round(3)

logger.info("Feature Importance as per Random Forest")
logger.info(feature_importance_df.to_string())
print(feature_importance_df.to_string())

                   Feature  importance_dt  importance_rf
35                 TotalSF          0.295          0.432
3              OverallQual          0.540          0.376
13                2ndFlrSF          0.031          0.025
5                YearBuilt          0.002          0.016
33                HouseAge          0.001          0.012
36               TotalBath          0.022          0.011
144               BsmtQual          0.000          0.010
15               GrLivArea          0.005          0.010
2                  LotArea          0.006          0.009
23              GarageArea          0.010          0.009
1              LotFrontage          0.020          0.009
8               BsmtFinSF1          0.010          0.009
150            KitchenQual          0.001          0.007
34           HouseRemodAge          0.001          0.006
6             YearRemodAdd          0.012          0.006
10               BsmtUnfSF          0.004          0.005
22              Fireplaces     

### XGBoost

In [399]:
from xgboost import XGBRegressor


categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

xgb_preprocessor = ColumnTransformer(transformers=[
    ('num', SimpleImputer(strategy='median'), num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal),
    #('pass', SimpleImputer(strategy='median'), pass_cols)
])

xgboost_regressor = Pipeline(
    steps=[
        ("pre", xgb_preprocessor),
        ("random_forest", XGBRegressor(
                eval_metric='rmse', 
                max_depth = 6, 
                n_estimators=500, 
                min_samples_split=10, 
                learning_rate=0.05, 
                random_state=42
                ))
    ]
)


xgboost_regressor.fit(X_train, y_train)
y_test_pred = xgboost_regressor.predict(X_test)
y_train_pred = xgboost_regressor.predict(X_train)


r2_score_test = r2_score(y_test,y_test_pred)
r2_score_train = r2_score(y_train, y_train_pred)

rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

r2_scores.append({
            "Model": "XGBoostRegressor", 
            "R2_score-Test": r2_score_test, 
            "R2_score-Train": r2_score_train,
            "RMSLE-Test": rmse_test, 
            "RMSLE-Train": rmse_train
            })  

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/xgboost/training.py:200: UserWarning: [20:11:32] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:782: 
Parameters: { "min_samples_split" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466,NaN,NaN
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786,NaN,NaN
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273,NaN,NaN
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910,NaN,NaN
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680,NaN,NaN
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795,NaN,NaN
6,Decision Tree,0.792529,0.934768,NaN,NaN,0.193743,0.102175
7,Random Forst Regressor,0.866345,0.935190,NaN,NaN,0.160718,0.104405
8,XGBoostRegressor,0.910951,0.999274,NaN,NaN,0.140071,0.014706


In [400]:
root_mean_squared_error(y_test, xgboost_regressor.predict(X_test))
from sklearn.metrics import root_mean_squared_log_error

root_mean_squared_log_error(y_test, xgboost_regressor.predict(X_test))

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


0.14007148146629333

### Hyperparameter tuning

In [401]:
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform

xgb_params ={
        "xgb__n_estimators": randint(300, 1000),
        "xgb__max_depth": randint(3, 10),
        "xgb__learning_rate": uniform(0.01, 0.2), 
        'xgb__subsample': uniform(0.6, 0.4),
        'xgb__colsample_bytree': uniform(0.6, 0.4),
        'xgb__reg_alpha': uniform(0, 1),
        'xgb__reg_lambda': uniform(0.5, 2)
    }

xgboost_regressor = Pipeline(
    steps=[
        ("pre", xgb_preprocessor),
        ("xgb", XGBRegressor(random_state=42))
    ]
)

random_search = RandomizedSearchCV(
    xgboost_regressor,
    param_distributions=xgb_params,
    n_iter=50,
    cv=5,
    scoring='r2',
    random_state=42,
    n_jobs=-1
)

random_search.fit(X_train, y_train)
print(f"Best param: {random_search.best_params_}")
print(f"Best CV accuracy: {random_search.best_score_}")

best_xgboost = random_search.best_estimator_

print(f"Best rmse: {root_mean_squared_log_error(y_test, best_xgboost.predict(X_test))}")

y_test_pred = best_xgboost.predict(X_test)
y_train_pred = best_xgboost.predict(X_train)


r2_score_test = r2_score(y_test,y_test_pred)
r2_score_train = r2_score(y_train, y_train_pred)

rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

r2_scores.append({
            "Model": "Best XGBoostRegressor", 
            "R2_score-Test": r2_score_test, 
            "R2_score-Train": r2_score_train,
            "RMSLE-Test": rmse_test, 
            "RMSLE-Train": rmse_train
            })  

    
    
pd.DataFrame(r2_scores)


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [2, 10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [4, 5, 6] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-

Best param: {'xgb__colsample_bytree': np.float64(0.6693458614031088), 'xgb__learning_rate': np.float64(0.08821212151464816), 'xgb__max_depth': 4, 'xgb__n_estimators': 687, 'xgb__reg_alpha': np.float64(0.31171107608941095), 'xgb__reg_lambda': np.float64(1.5401360423556216), 'xgb__subsample': np.float64(0.8186841117373118)}
Best CV accuracy: 0.8794867396354675
Best rmse: 0.13787983357906342


/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466,NaN,NaN
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786,NaN,NaN
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273,NaN,NaN
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910,NaN,NaN
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680,NaN,NaN
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795,NaN,NaN
6,Decision Tree,0.792529,0.934768,NaN,NaN,0.193743,0.102175
7,Random Forst Regressor,0.866345,0.935190,NaN,NaN,0.160718,0.104405
8,XGBoostRegressor,0.910951,0.999274,NaN,NaN,0.140071,0.014706
9,Best XGBoostRegressor,0.911734,0.999308,NaN,NaN,0.137880,0.014105


### LightGBM

In [402]:
import lightgbm as lgb


numeric_transformer = Pipeline(
    steps=[
        ('mean_imputer', SimpleImputer(strategy='mean'))
        ]
)

categorical_onehot_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("onehotencoder", OneHotEncoder(drop="first", handle_unknown='ignore', sparse_output=False))
    ]
)

categorical_ordinal_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='constant', fill_value='NA')),
        ("ordinal", OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
    ]
)

lgb_preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, num_cols),
    ('cat_onehotencode', categorical_onehot_transformer, cat_onehotencode),
    ('cat_ordinal', categorical_ordinal_transformer, cat_ordinal),
    #('pass', SimpleImputer(strategy='median'), pass_cols)
])


lgb_regressor = Pipeline(
    steps=[
        ("pre", lgb_preprocessor),
        ("LGBM Regressor", lgb.LGBMRegressor(
                eval_metric='rmse', 
                max_depth = 6, 
                n_estimators=500, 
                min_samples_split=10, 
                learning_rate=0.05, 
                random_state=42
                ))
    ]
)


lgb_regressor.fit(X_train, y_train)
y_test_pred = lgb_regressor.predict(X_test)
y_train_pred = lgb_regressor.predict(X_train)


r2_score_test = r2_score(y_test,y_test_pred)
r2_score_train = r2_score(y_train, y_train_pred)

rmse_test = root_mean_squared_log_error(y_test,y_test_pred)
rmse_train = root_mean_squared_log_error(y_train, y_train_pred)  

r2_scores.append({
            "Model": "lgb_regressor", 
            "R2_score-Test": r2_score_test, 
            "R2_score-Train": r2_score_train,
            "RMSLE-Test": rmse_test, 
            "RMSLE-Train": rmse_train
            })  

    
    
pd.DataFrame(r2_scores)

[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Warning] Unknown parameter: eval_metric
[LightGBM] [Warning] Unknown parameter: min_samples_split
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002571 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3184
[LightGBM] [Info] Number of data points in the train set: 1168, number of used features: 103
[LightGBM] [Info] Start training from score 181441.541952
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain:

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [10, 12] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/utils/validation.py:2691: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


,Model,R2_score-Test,R2_score-Train,RMSE-Test,RMSE-Train,RMSLE-Test,RMSLE-Train
0,linear_regression,0.707889,0.915477,47334.875151,22453.058466,NaN,NaN
1,linear_regression_pca,0.837451,0.829092,35310.120620,31927.901786,NaN,NaN
2,linear_regression_ridge,0.851399,0.906452,33761.220246,23621.453273,NaN,NaN
3,linear_regression_ridge_pca,0.837452,0.829092,35309.952107,31927.901910,NaN,NaN
4,linear_regression_lasso,0.708275,0.915477,47303.575067,22453.106680,NaN,NaN
5,linear_regression_lasso_pca,0.837451,0.829092,35310.129978,31927.901795,NaN,NaN
6,Decision Tree,0.792529,0.934768,NaN,NaN,0.193743,0.102175
7,Random Forst Regressor,0.866345,0.935190,NaN,NaN,0.160718,0.104405
8,XGBoostRegressor,0.910951,0.999274,NaN,NaN,0.140071,0.014706
9,Best XGBoostRegressor,0.911734,0.999308,NaN,NaN,0.137880,0.014105


## Saving and loading model using joblib

In [403]:
test_df = pd.read_csv('test.csv')
test_df.shape

(1459, 80)

In [404]:
test_df_id = test_df.drop(["Id"], axis=1)

In [405]:
test_df_cleaned = data_cleanup(test_df_id)

In [406]:
# test_df_processed = xgb_preprocessor.transform(test_df_cleaned)
# test_df_processed.shape

In [407]:
import joblib

joblib.dump(best_xgboost, 'best_model.joblib')

['best_model.joblib']

In [ ]:
test_df_cleaned.head()

In [408]:
loaded_model = joblib.load('best_model.joblib')
test_pred = loaded_model.predict(test_df_cleaned)
test_pred

/Users/sumana/Desktop/DataScience-AIML-2025/DataScience-AI-ML-2025-Sumana/.venv/lib/python3.13/site-packages/sklearn/preprocessing/_encoders.py:261: UserWarning: Found unknown categories in columns [0, 2, 14] during transform. These unknown categories will be encoded as all zeros
  warnings.warn(msg, UserWarning)


array([131960.39, 166574.62, 187916.92, ..., 177092.81, 125943.47,
       219439.61], shape=(1459,), dtype=float32)

In [409]:
test_pred_df = pd.DataFrame({"Id": test_df["Id"], "SalePrice": test_pred})

In [410]:
test_pred_df

,Id,SalePrice
0,1461,131960.390625
1,1462,166574.625000
2,1463,187916.921875
3,1464,187217.609375
4,1465,187793.250000
...,...,...
1454,2915,85102.570312
1455,2916,80838.078125
1456,2917,177092.812500
1457,2918,125943.468750


In [411]:
test_pred_df.to_csv("submission.csv", index=False)

Find number od days since last sell, Once it is calculated drop "MoSold", "YrSold", "DateSold" Columns

In [412]:
# house["DateSold"] = house.apply(lambda row: f"{row['YrSold']}-{row['MoSold']}-01", axis=1)
# house["DateSold"] = pd.to_datetime(house["DateSold"])
# house.head()

In [413]:
# house["DaysSinceLastSell"] = (pd.Timestamp.today() - house["DateSold"]).dt.days
# house.head()

In [414]:
# house.drop(["MoSold", "YrSold", "DateSold"], axis=1, inplace=True)
# house.head()

In [415]:
# house["Age"] = pd.Timestamp.today().year - house["YearBuilt"]

In [416]:
# house.drop(["YearBuilt"],axis=1, inplace=True)
